# TpMr 的 THM（C 发射 → D 接收，按图示展开，兼容旧版 Python 内核）

本 notebook 按 **TpMr = T 星发射到 M 星** 处理，并映射为 **C 星发射、D 星接收**。

采用图示展开方式：
- 接收点取 `r_r = r_D(t_r)`，不做时间展开
- 发射点取 `r_e = r_C(t_r - Δt_emit)`，并做二阶泰勒展开
- 发射端加速度按地球点质量模型 `a = GM / R^2` 计算
- 其余静力势 `W_HM` 的求解逻辑保持原程序不变

这版已改成**兼容旧版 Python 内核**：
- 去掉了 `str | Path` 这一类 3.10+ 才支持的类型写法
- 可以在较旧的 Jupyter / Python 内核中运行


并按你更正后的方向定义：
- `d0 = (r_D - r_C) / |r_D - r_C|`


并且 `T_HM` 的光路积分也改成与 MeTp 最新版本一致，按论文式 (4.21) 的**等步长分段梯形求和**进行计算。

最终输出结果已精简为两列：`gps_time` 和 `THM`。


In [1]:
CONFIG = {
    # 输入文件
    "c_file": r"GNI1B_2022-06-05_C_04.txt",   # C 星 = T 星 = 发射端
    "d_file": r"GNI1B_2022-06-05_D_04.txt",   # D 星 = M 星 = 接收端
    "gfc_file": r"EIGEN-6C4.gfc",             # 静力场模型

    # THM 计算参数（其余 WHM 逻辑保持原程序不变）
    "lmax": 2,
    "n_path": 10,
    "n_potential_path": 15,
    "r_ref_factor": 50.0,

    # 只计算前若干条记录；None 表示全算
    "max_rows": None,

    # 是否显示进度条
    "show_progress": True,

    # 输出文件
    "out_xlsx": "TpMr_THM_C_emit_D_recv_image_expand_d0_D_minus_C_eq421_pyold_minimal.xlsx",
}

CONFIG

{'c_file': 'GNI1B_2022-06-05_C_04.txt',
 'd_file': 'GNI1B_2022-06-05_D_04.txt',
 'gfc_file': 'EIGEN-6C4.gfc',
 'lmax': 2,
 'n_path': 10,
 'n_potential_path': 15,
 'r_ref_factor': 50.0,
 'max_rows': None,
 'show_progress': True,
 'out_xlsx': 'TpMr_THM_C_emit_D_recv_image_expand_d0_D_minus_C_eq421_pyold_minimal.xlsx'}

In [2]:
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid
from scipy.special import gammaln, lpmv

import astropy.units as u
from astropy.coordinates import CartesianRepresentation, GCRS, ITRS
from astropy.time import Time, TimeDelta
from astropy.utils import iers

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

try:
    iers.conf.auto_download = True
except Exception:
    pass

In [3]:
C0 = 299_792_458.0
GNI_GPS_TIME_EPOCH_UTC = Time("2000-01-01T11:59:47", scale="utc")


@dataclass
class GravityModel:
    GM: float
    Re: float
    C: np.ndarray
    S: np.ndarray
    lmax_use: int
    signed_norm_table: np.ndarray
    m_index: np.ndarray


@dataclass
class OneWayIterationResult:
    T_HM: float
    n_iter: int
    last_abs_update: float


def norm3(x: np.ndarray) -> float:
    return float(np.linalg.norm(x))


def unit3(x: np.ndarray, eps: float = 1e-30) -> np.ndarray:
    n = norm3(x)
    if n < eps:
        raise ValueError("向量范数过小，无法单位化。")
    return np.asarray(x, dtype=float) / n


def direct_emit_position_from_velocity(r_tr: np.ndarray,
                                       v_tr: np.ndarray,
                                       delta_t_inst: float,
                                       d0_inst: np.ndarray) -> np.ndarray:
    r_tr = np.asarray(r_tr, dtype=float)
    v_tr = np.asarray(v_tr, dtype=float)
    d0_inst = np.asarray(d0_inst, dtype=float)
    dt_emit = float(delta_t_inst + delta_t_inst * np.dot(d0_inst, v_tr) / C0)
    return r_tr - v_tr * dt_emit


def gps_seconds_to_astropy_time(gps_seconds_from_2000_epoch: float) -> Time:
    return GNI_GPS_TIME_EPOCH_UTC + TimeDelta(float(gps_seconds_from_2000_epoch), format="sec")


def read_gni1b_txt(path: Union[str, Path]) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到文件: {path}")

    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    try:
        end_idx = next(i for i, line in enumerate(lines) if line.strip() == "# End of YAML header")
    except StopIteration as exc:
        raise ValueError(f"文件 {path} 未找到 '# End of YAML header' 标记。") from exc

    col_names = [
        "gps_time", "sat_id", "coord_ref",
        "xpos", "ypos", "zpos",
        "xpos_err", "ypos_err", "zpos_err",
        "xvel", "yvel", "zvel",
        "xvel_err", "yvel_err", "zvel_err",
        "qualflg",
    ]

    df = pd.read_csv(
        path,
        sep=r"\s+",
        skiprows=end_idx + 1,
        header=None,
        names=col_names,
        engine="python",
    )

    df = df[["gps_time", "sat_id", "coord_ref", "xpos", "ypos", "zpos", "xvel", "yvel", "zvel"]].copy()
    if not (df["coord_ref"] == "I").all():
        raise ValueError(f"文件 {path} 含非惯性系记录，coord_ref 应为 'I'。")
    return df.reset_index(drop=True)


def point_mass_gravity_acceleration(r_gcrs_m: np.ndarray, GM: float) -> np.ndarray:
    """
    地球点质量模型：
        a = - GM / R^3 * r
    其模长满足：
        |a| = GM / R^2
    """
    r_gcrs_m = np.asarray(r_gcrs_m, dtype=float)
    r_norm = norm3(r_gcrs_m)
    if r_norm == 0.0:
        raise ValueError("位置向量模长为零，无法计算点质量引力加速度。")
    return -(float(GM) / r_norm**3) * r_gcrs_m


def prepare_satellite_dataframe(path: Union[str, Path], GM_earth: float) -> pd.DataFrame:
    """
    读取轨道文件，并按地球点质量模型计算加速度：
        a = GM / R^2
    在程序中使用向量形式：
        a_vec = -GM / R^3 * r
    """
    out = read_gni1b_txt(path).copy()
    r_all = out[["xpos", "ypos", "zpos"]].to_numpy(dtype=float)
    a_all = np.apply_along_axis(point_mass_gravity_acceleration, 1, r_all, float(GM_earth))
    out["xacc"] = a_all[:, 0]
    out["yacc"] = a_all[:, 1]
    out["zacc"] = a_all[:, 2]
    return out



def build_signed_norm_table(lmax: int) -> np.ndarray:
    signed_norm = np.zeros((lmax + 1, lmax + 1), dtype=float)
    for l in range(lmax + 1):
        for m in range(l + 1):
            delta_m0 = 1.0 if m == 0 else 0.0
            logN = 0.5 * (
                np.log((2.0 - delta_m0) * (2 * l + 1))
                + gammaln(l - m + 1)
                - gammaln(l + m + 1)
            )
            signed_norm[l, m] = ((-1.0) ** m) * np.exp(logN)
    return signed_norm


def load_icgem_gfc(path: Union[str, Path], lmax_use: int) -> GravityModel:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到 gfc 文件: {path}")

    GM = None
    Re = None
    max_degree_file = None
    norm_type = None
    coeffs: List[Tuple[int, int, float, float]] = []

    in_header = True
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if in_header:
                if line.startswith("end_of_head"):
                    in_header = False
                    continue
                parts = line.split()
                key = parts[0]
                if key == "earth_gravity_constant":
                    GM = float(parts[1])
                elif key == "radius":
                    Re = float(parts[1])
                elif key == "max_degree":
                    max_degree_file = int(parts[1])
                elif key == "norm":
                    norm_type = parts[1]
                continue

            if line.startswith("gfc"):
                parts = line.split()
                l = int(parts[1])
                m = int(parts[2])
                if l <= lmax_use:
                    coeffs.append((l, m, float(parts[3]), float(parts[4])))

    if GM is None or Re is None or max_degree_file is None:
        raise ValueError("gfc 文件头读取失败，请检查文件格式。")
    if lmax_use > max_degree_file:
        raise ValueError(f"lmax_use={lmax_use} 超过文件最大阶数 {max_degree_file}")
    if str(norm_type).lower() != "fully_normalized":
        raise ValueError(f"当前代码按 fully_normalized 系数实现，文件 norm={norm_type} 需匹配。")

    C = np.zeros((lmax_use + 1, lmax_use + 1), dtype=float)
    S = np.zeros((lmax_use + 1, lmax_use + 1), dtype=float)
    for l, m, c, s in coeffs:
        C[l, m] = c
        S[l, m] = s

    return GravityModel(
        GM=GM,
        Re=Re,
        C=C,
        S=S,
        lmax_use=lmax_use,
        signed_norm_table=build_signed_norm_table(lmax_use),
        m_index=np.arange(lmax_use + 1, dtype=float),
    )


@lru_cache(maxsize=4096)
def gcrs_to_itrs_rotation_matrix(gps_time_s: float) -> np.ndarray:
    gps_time_s = float(gps_time_s)
    obstime = gps_seconds_to_astropy_time(gps_time_s)
    basis = np.eye(3, dtype=float)
    mat = np.empty((3, 3), dtype=float)
    for j in range(3):
        rep = CartesianRepresentation(x=basis[0, j] * u.m, y=basis[1, j] * u.m, z=basis[2, j] * u.m)
        gcrs = GCRS(rep, obstime=obstime)
        itrs = gcrs.transform_to(ITRS(obstime=obstime))
        mat[:, j] = np.array(itrs.cartesian.xyz.to_value(u.m), dtype=float)
    return mat


def itrs_to_gcrs_rotation_matrix(gps_time_s: float) -> np.ndarray:
    return gcrs_to_itrs_rotation_matrix(float(gps_time_s)).T


def gcrs_to_itrs_vec(r_gcrs_m: np.ndarray, gps_time_s: float) -> np.ndarray:
    return gcrs_to_itrs_rotation_matrix(float(gps_time_s)) @ np.asarray(r_gcrs_m, dtype=float)


def cartesian_to_spherical(r_xyz: np.ndarray) -> Tuple[float, float, float]:
    x, y, z = np.asarray(r_xyz, dtype=float)
    r = norm3(r_xyz)
    if r == 0.0:
        raise ValueError("位置向量为零，无法转球坐标。")
    theta = np.arccos(np.clip(z / r, -1.0, 1.0))
    lamb = np.arctan2(y, x)
    return r, theta, lamb


def fully_normalized_plm(l: int, m: int, x: float, signed_norm_table: np.ndarray) -> float:
    return float(signed_norm_table[l, m] * lpmv(m, l, x))


def build_fully_normalized_plm_tables(lmax: int,
                                      x: float,
                                      theta: float,
                                      signed_norm_table: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    pbar = np.zeros((lmax + 1, lmax + 1), dtype=float)
    dpbar_dtheta = np.zeros((lmax + 1, lmax + 1), dtype=float)

    for l in range(lmax + 1):
        m_arr = np.arange(l + 1)
        pbar[l, :l + 1] = signed_norm_table[l, :l + 1] * lpmv(m_arr, l, x)

    sin_theta = float(np.sin(theta))
    if abs(sin_theta) < 1e-15:
        sin_theta = 1e-15 if sin_theta >= 0.0 else -1e-15

    for l in range(1, lmax + 1):
        for m in range(l + 1):
            if l - 1 >= m:
                coeff = np.sqrt(((2.0 * l + 1.0) / (2.0 * l - 1.0)) * (l * l - m * m))
                pbar_lm1 = pbar[l - 1, m]
            else:
                coeff = 0.0
                pbar_lm1 = 0.0
            dpbar_dtheta[l, m] = (l * x * pbar[l, m] - coeff * pbar_lm1) / sin_theta

    return pbar, dpbar_dtheta


def compute_static_whm_from_itrs(r_itrs_m: np.ndarray, gravity: GravityModel) -> float:
    r, theta, lamb = cartesian_to_spherical(r_itrs_m)
    x = np.cos(theta)

    pbar, _ = build_fully_normalized_plm_tables(
        gravity.lmax_use, x, theta, gravity.signed_norm_table
    )

    m_index = gravity.m_index
    cos_m_lambda = np.cos(m_index * lamb)
    sin_m_lambda = np.sin(m_index * lamb)

    q = gravity.Re / r
    radial = q * q
    total = 0.0

    for l in range(1, gravity.lmax_use + 1):
        trig = gravity.C[l, :l + 1] * cos_m_lambda[:l + 1] + gravity.S[l, :l + 1] * sin_m_lambda[:l + 1]
        total += radial * float(np.dot(trig, pbar[l, :l + 1]))
        radial *= q

    return gravity.GM / gravity.Re * total


def compute_static_ahm_from_itrs(r_itrs_m: np.ndarray, gravity: GravityModel) -> np.ndarray:
    r, theta, lamb = cartesian_to_spherical(r_itrs_m)
    x = np.cos(theta)
    sin_theta = float(np.sin(theta))
    if abs(sin_theta) < 1e-15:
        sin_theta = 1e-15 if sin_theta >= 0.0 else -1e-15

    pbar, dpbar_dtheta = build_fully_normalized_plm_tables(
        gravity.lmax_use, x, theta, gravity.signed_norm_table
    )

    m_index = gravity.m_index
    cos_m_lambda = np.cos(m_index * lamb)
    sin_m_lambda = np.sin(m_index * lamb)

    dW_dr = 0.0
    dW_dtheta = 0.0
    dW_dlambda = 0.0

    q = gravity.Re / r
    radial = q * q

    for l in range(1, gravity.lmax_use + 1):
        pref = gravity.GM / gravity.Re * radial

        trig = gravity.C[l, :l + 1] * cos_m_lambda[:l + 1] + gravity.S[l, :l + 1] * sin_m_lambda[:l + 1]
        dtrig_dlambda = m_index[:l + 1] * (
            -gravity.C[l, :l + 1] * sin_m_lambda[:l + 1] + gravity.S[l, :l + 1] * cos_m_lambda[:l + 1]
        )

        pbar_l = pbar[l, :l + 1]
        dpbar_l = dpbar_dtheta[l, :l + 1]

        dW_dr += pref * (-(l + 1) / r) * float(np.dot(trig, pbar_l))
        dW_dtheta += pref * float(np.dot(trig, dpbar_l))
        dW_dlambda += pref * float(np.dot(dtrig_dlambda, pbar_l))

        radial *= q

    a_r = dW_dr
    a_theta = dW_dtheta / r
    a_lambda = dW_dlambda / (r * sin_theta)

    st = np.sin(theta)
    ct = np.cos(theta)
    cl = np.cos(lamb)
    sl = np.sin(lamb)

    e_r = np.array([st * cl, st * sl, ct], dtype=float)
    e_theta = np.array([ct * cl, ct * sl, -st], dtype=float)
    e_lambda = np.array([-sl, cl, 0.0], dtype=float)

    return a_r * e_r + a_theta * e_theta + a_lambda * e_lambda


def evaluate_static_ahm_at_gcrs(gps_time_s: float, r_gcrs_m: np.ndarray, gravity: GravityModel) -> np.ndarray:
    rot_g2i = gcrs_to_itrs_rotation_matrix(float(gps_time_s))
    r_itrs_m = rot_g2i @ np.asarray(r_gcrs_m, dtype=float)
    a_itrs_mps2 = compute_static_ahm_from_itrs(r_itrs_m, gravity)
    return rot_g2i.T @ a_itrs_mps2


def compute_static_whm_from_gcrs_line_integral(gps_time_s: float,
                                               r_gcrs_m: np.ndarray,
                                               gravity: GravityModel,
                                               n_potential_path: int = 64,
                                               r_ref_factor: float = 50.0) -> float:
    if n_potential_path < 2:
        raise ValueError("n_potential_path 至少应为 2。")
    if r_ref_factor <= 1.0:
        raise ValueError("r_ref_factor 应大于 1。")

    gps_time_s = float(gps_time_s)
    r_gcrs_m = np.asarray(r_gcrs_m, dtype=float)
    r0 = norm3(r_gcrs_m)
    u_hat = unit3(r_gcrs_m)
    r_ref = max(float(r_ref_factor) * gravity.Re, r0 * (1.0 + 1e-12))

    rho_nodes = np.geomspace(r0, r_ref, int(n_potential_path) + 1)
    a_dot_dr = np.empty_like(rho_nodes)

    rot_g2i = gcrs_to_itrs_rotation_matrix(gps_time_s)
    u_hat_itrs = rot_g2i @ u_hat

    for i, rho in enumerate(rho_nodes):
        point_i_itrs = u_hat_itrs * rho
        a_itrs_i = compute_static_ahm_from_itrs(point_i_itrs, gravity)
        a_dot_dr[i] = float(np.dot(a_itrs_i, u_hat_itrs))

    r_ref_itrs = u_hat_itrs * r_ref
    w_ref = compute_static_whm_from_itrs(r_ref_itrs, gravity)
    return float(w_ref - trapezoid(a_dot_dr, x=rho_nodes))


def evaluate_whm_at_gcrs(gps_time_s: float,
                         r_gcrs_m: np.ndarray,
                         gravity: GravityModel,
                         n_potential_path: int = 64,
                         r_ref_factor: float = 50.0) -> float:
    return compute_static_whm_from_gcrs_line_integral(
        gps_time_s=float(gps_time_s),
        r_gcrs_m=r_gcrs_m,
        gravity=gravity,
        n_potential_path=n_potential_path,
        r_ref_factor=r_ref_factor,
    )


def compute_thm(te_seconds: float,
                delta_t_sr: float,
                re_gcrs: np.ndarray,
                rr_gcrs: np.ndarray,
                gravity: GravityModel,
                n_path: int = 16,
                n_potential_path: int = 64,
                r_ref_factor: float = 50.0) -> float:
    """
    按论文式 (4.21) 的等步长分段梯形求和计算 T_HM：
        T_HM ≈ 2 * Δt_SR / (c0^2 * N)
               * [ sum_{n=1}^{N-1} W_n + (W_0 + W_N)/2 ]
    """
    if n_path < 1:
        raise ValueError("n_path 至少应为 1。")

    N = int(n_path)

    lam = np.linspace(0.0, 1.0, N + 1)
    t_nodes = te_seconds + delta_t_sr * lam
    r_nodes = re_gcrs[None, :] + (rr_gcrs - re_gcrs)[None, :] * lam[:, None]

    w_vals = np.empty(N + 1, dtype=float)
    for i in range(N + 1):
        w_vals[i] = evaluate_whm_at_gcrs(
            gps_time_s=float(t_nodes[i]),
            r_gcrs_m=r_nodes[i],
            gravity=gravity,
            n_potential_path=n_potential_path,
            r_ref_factor=r_ref_factor,
        )

    if N == 1:
        weighted_sum = 0.5 * (float(w_vals[0]) + float(w_vals[1]))
    else:
        weighted_sum = float(np.sum(w_vals[1:N])) + 0.5 * (float(w_vals[0]) + float(w_vals[N]))

    return 2.0 * float(delta_t_sr) / (C0**2 * N) * weighted_sum




@dataclass
class OneWayAnalyticResult:
    T_HM: float
    dt_inst: float
    dt_emit: float
    dt_recv_corr: float
    te_seconds: float
    delta_t_sr: float
    re_gcrs: np.ndarray
    rr_gcrs: np.ndarray
    d0: np.ndarray
    d0_dot_vC: float
    d0_dot_vD: float
    aC_tr: np.ndarray
    aD_tr: np.ndarray


def build_tpmr_c_emit_d_recv_geometry_image_expand(rC_tr, vC_tr, rD_tr, vD_tr, GM_earth, c=C0):
    """
    TpMr 的图示展开方式：
        r_r = r_D(t_r)
        r_e = r_C(t_r - dt_emit)

    其中
        d0      = (r_C - r_D) / |r_C - r_D|
        dt_inst = |r_C - r_D| / c
        dt_emit = dt_inst + dt_inst * (d0·v_C) / c

    发射点按二阶泰勒展开：
        r_e ≈ r_C(t_r) - v_C(t_r) * dt_emit + 0.5 * a_C(t_r) * dt_emit^2

    接收点不展开：
        r_r = r_D(t_r)

    方向向量按你更正后的定义：
        d0 = (r_D - r_C) / |r_D - r_C|

    加速度采用地球点质量模型。
    """
    rC_tr = np.asarray(rC_tr, dtype=float)
    vC_tr = np.asarray(vC_tr, dtype=float)
    rD_tr = np.asarray(rD_tr, dtype=float)
    vD_tr = np.asarray(vD_tr, dtype=float)

    dr_CD = rD_tr - rC_tr
    rho0 = norm3(dr_CD)
    if rho0 == 0.0:
        raise ValueError("rC_tr 与 rD_tr 重合，无法定义传播方向 d0。")

    d0 = dr_CD / rho0
    dt_inst = rho0 / c

    d0_dot_vC = float(np.dot(d0, vC_tr))
    d0_dot_vD = float(np.dot(d0, vD_tr))

    dt_emit = dt_inst * (1.0 + d0_dot_vC / c)
    dt_recv_corr = 0.0

    aC_tr = point_mass_gravity_acceleration(rC_tr, GM=GM_earth)
    aD_tr = point_mass_gravity_acceleration(rD_tr, GM=GM_earth)

    rr_gcrs = rD_tr
    re_gcrs = rC_tr - vC_tr * dt_emit + 0.5 * aC_tr * (dt_emit ** 2)

    return {
        "re_gcrs": re_gcrs,
        "rr_gcrs": rr_gcrs,
        "dt_inst": float(dt_inst),
        "dt_emit": float(dt_emit),
        "dt_recv_corr": float(dt_recv_corr),
        "d0": d0,
        "d0_dot_vC": float(d0_dot_vC),
        "d0_dot_vD": float(d0_dot_vD),
        "aC_tr": aC_tr,
        "aD_tr": aD_tr,
    }


def compute_one_way_thm_analytic(tr_seconds,
                                 rC_tr,
                                 vC_tr,
                                 rD_tr,
                                 vD_tr,
                                 gravity,
                                 n_path=16,
                                 n_potential_path=64,
                                 r_ref_factor=50.0):
    geo = build_tpmr_c_emit_d_recv_geometry_image_expand(
        rC_tr=rC_tr,
        vC_tr=vC_tr,
        rD_tr=rD_tr,
        vD_tr=vD_tr,
        GM_earth=gravity.GM,
        c=C0,
    )

    re_gcrs = np.asarray(geo["re_gcrs"], dtype=float)
    rr_gcrs = np.asarray(geo["rr_gcrs"], dtype=float)
    delta_t_sr = norm3(rr_gcrs - re_gcrs) / C0
    te_seconds = float(tr_seconds) - float(delta_t_sr)

    thm = compute_thm(
        te_seconds=te_seconds,
        delta_t_sr=delta_t_sr,
        re_gcrs=re_gcrs,
        rr_gcrs=rr_gcrs,
        gravity=gravity,
        n_path=n_path,
        n_potential_path=n_potential_path,
        r_ref_factor=r_ref_factor,
    )

    return OneWayAnalyticResult(
        T_HM=float(thm),
        dt_inst=float(geo["dt_inst"]),
        dt_emit=float(geo["dt_emit"]),
        dt_recv_corr=float(geo["dt_recv_corr"]),
        te_seconds=float(te_seconds),
        delta_t_sr=float(delta_t_sr),
        re_gcrs=re_gcrs,
        rr_gcrs=rr_gcrs,
        d0=np.asarray(geo["d0"], dtype=float),
        d0_dot_vC=float(geo["d0_dot_vC"]),
        d0_dot_vD=float(geo["d0_dot_vD"]),
        aC_tr=np.asarray(geo["aC_tr"], dtype=float),
        aD_tr=np.asarray(geo["aD_tr"], dtype=float),
    )


def compute_tpmr_thm_c_emit_d_recv_image_expand(cfg):
    gravity = load_icgem_gfc(str(cfg["gfc_file"]), lmax_use=int(cfg.get("lmax", 60)))
    c_df = prepare_satellite_dataframe(str(cfg["c_file"]), GM_earth=gravity.GM)
    d_df = prepare_satellite_dataframe(str(cfg["d_file"]), GM_earth=gravity.GM)

    merged = pd.merge(
        c_df,
        d_df,
        on="gps_time",
        suffixes=("_tx", "_rx"),
        how="inner",
    ).sort_values("gps_time").reset_index(drop=True)

    max_rows = cfg.get("max_rows", None)
    if max_rows not in [None, "", "None"]:
        merged = merged.iloc[:int(max_rows)].copy()

    records = merged[[
        "gps_time",
        "xpos_tx", "ypos_tx", "zpos_tx",
        "xvel_tx", "yvel_tx", "zvel_tx",
        "xpos_rx", "ypos_rx", "zpos_rx",
        "xvel_rx", "yvel_rx", "zvel_rx",
    ]].itertuples(index=False, name=None)

    total_rows = len(merged)
    show_progress = bool(cfg.get("show_progress", True))
    if show_progress and tqdm is not None:
        records = tqdm(records, total=total_rows, desc="C → D THM 图示展开计算进度（d0 = D - C）")
    elif show_progress:
        print("C → D THM 图示展开计算开始（d0 = D - C） ...")

    n_path = int(cfg.get("n_path", 16))
    n_potential_path = int(cfg.get("n_potential_path", 64))
    r_ref_factor = float(cfg.get("r_ref_factor", 50.0))

    rows = []
    append_row = rows.append

    for (
        gps_time,
        xpos_tx, ypos_tx, zpos_tx,
        xvel_tx, yvel_tx, zvel_tx,
        xpos_rx, ypos_rx, zpos_rx,
        xvel_rx, yvel_rx, zvel_rx,
    ) in records:
        result = compute_one_way_thm_analytic(
            tr_seconds=float(gps_time),
            rC_tr=np.array([xpos_tx, ypos_tx, zpos_tx], dtype=float),
            vC_tr=np.array([xvel_tx, yvel_tx, zvel_tx], dtype=float),
            rD_tr=np.array([xpos_rx, ypos_rx, zpos_rx], dtype=float),
            vD_tr=np.array([xvel_rx, yvel_rx, zvel_rx], dtype=float),
            gravity=gravity,
            n_path=n_path,
            n_potential_path=n_potential_path,
            r_ref_factor=r_ref_factor,
        )

        append_row({
            "gps_time": float(gps_time),
            "THM": float(result.T_HM),
        })

    return pd.DataFrame(rows)


In [4]:
RESULT_DF = compute_tpmr_thm_c_emit_d_recv_image_expand(CONFIG)
RESULT_DF.to_excel(CONFIG["out_xlsx"], sheet_name="THM_only", index=False)

print(f"结果已写出: {CONFIG['out_xlsx']}")
print(RESULT_DF.head())


C → D THM 图示展开计算进度（d0 = D - C）:   0%|                                                                                              | 71/86400 [00:06<2:15:05, 10.65it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

df_check = pd.read_excel(CONFIG["out_xlsx"])
print(df_check.shape)
print(df_check.head())
